In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torchvision.models import resnet18
from torch.utils.data import DataLoader
import numpy as np
import random
import pandas as pd
import time
import scipy.stats as stats

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

Device: cuda


In [2]:
class TwoCropsTransform:
    def __init__(self, base_transform):
        self.base_transform = base_transform
    def __call__(self, x):
        return self.base_transform(x), self.base_transform(x)

ssl_base_transform = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2,1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.4,0.4,0.4,0.1),
    T.RandomGrayscale(p=0.2),
    T.ToTensor()
])

ssl_transform = TwoCropsTransform(ssl_base_transform)

train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=ssl_transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=T.ToTensor()
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

100%|██████████| 170M/170M [05:17<00:00, 537kB/s] 


In [3]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = resnet18(weights=None)
        backbone.fc = nn.Identity()
        self.backbone = backbone
    def forward(self, x):
        return self.backbone(x)

In [4]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim=2048, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim)
        )
    def forward(self, x):
        return self.net(x)

class BYOL(nn.Module):
    def __init__(self):
        super().__init__()
        self.online_encoder = Encoder()
        self.online_projector = MLP(512)
        self.online_predictor = MLP(256, 512, 256)

        self.target_encoder = Encoder()
        self.target_projector = MLP(512)

        for p in self.target_encoder.parameters():
            p.requires_grad = False
        for p in self.target_projector.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def update_target(self, tau=0.996):
        for o, t in zip(self.online_encoder.parameters(),
                        self.target_encoder.parameters()):
            t.data = tau * t.data + (1 - tau) * o.data

    def forward(self, x1, x2):
        o1 = self.online_predictor(
            self.online_projector(self.online_encoder(x1))
        )

        with torch.no_grad():
            t2 = self.target_projector(
                self.target_encoder(x2)
            )

        loss = 2 - 2 * F.cosine_similarity(o1, t2.detach(), dim=-1)
        return loss.mean()

In [5]:
class SimCLR(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.projector = MLP(512)

    def forward(self, x):
        z = self.projector(self.encoder(x))
        return F.normalize(z, dim=1)

def nt_xent_loss(z1, z2, temperature=0.5):
    N = z1.size(0)
    z = torch.cat([z1, z2], dim=0)

    sim = torch.mm(z, z.t()) / temperature
    mask = torch.eye(2*N, device=z.device).bool()
    sim.masked_fill_(mask, -9e15)

    positives = torch.cat([
        torch.arange(N,2*N),
        torch.arange(0,N)
    ]).to(z.device)

    return F.cross_entropy(sim, positives)

In [6]:
def jacobian_penalty(model, x):
    x = x.clone().detach().requires_grad_(True)
    z = model(x)

    v = torch.randn_like(z)

    Jv = torch.autograd.grad(
        outputs=z,
        inputs=x,
        grad_outputs=v,
        create_graph=True
    )[0]

    return Jv.pow(2).sum(dim=[1,2,3]).mean()

In [7]:
def train_byol(model, epochs=100):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

    total_start = time.time()

    for epoch in range(epochs):
        epoch_start = time.time()
        model.train()
        total_loss = 0

        for (x1,x2), _ in train_loader:
            x1,x2 = x1.to(device), x2.to(device)

            loss = model(x1,x2)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            model.update_target()

            total_loss += loss.item()

        epoch_time = (time.time() - epoch_start)/60
        total_time = (time.time() - total_start)/60

        print(f"[BYOL] {epoch+1:03d}/{epochs} | "
              f"Loss: {total_loss/len(train_loader):.4f} | "
              f"Epoch Time: {epoch_time:.2f}m | "
              f"Total Time: {total_time:.2f}m")

    return model.online_encoder


def train_simclr(model, epochs=100, lambda_jac=0.0):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

    total_start = time.time()

    for epoch in range(epochs):
        epoch_start = time.time()
        model.train()
        total_loss = 0

        for (x1,x2), _ in train_loader:
            x1,x2 = x1.to(device), x2.to(device)

            z1 = model(x1)
            z2 = model(x2)
            loss = nt_xent_loss(z1,z2)

            if lambda_jac > 0:
                jac = jacobian_penalty(model, x1)
                loss = loss + lambda_jac * jac

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        epoch_time = (time.time() - epoch_start)/60
        total_time = (time.time() - total_start)/60

        print(f"[SimCLR] {epoch+1:03d}/{epochs} | "
              f"Loss: {total_loss/len(train_loader):.4f} | "
              f"Epoch Time: {epoch_time:.2f}m | "
              f"Total Time: {total_time:.2f}m")

    return model.encoder

In [8]:
# =========================================================
# Linear Probe
# =========================================================

class LinearProbe(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        for p in self.encoder.parameters():
            p.requires_grad = False
        self.fc = nn.Linear(512, 10)

    def forward(self, x):
        with torch.no_grad():
            feat = self.encoder(x)
        return self.fc(feat)


def train_probe(encoder, train_loader, test_loader, epochs=10):
    model = LinearProbe(encoder).to(device)
    optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            logits = model(x)
            loss = criterion(logits, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

In [9]:
# =========================================================
# Metrics
# =========================================================

tensor_aug = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.4,0.4,0.4,0.1),
    T.RandomGrayscale(p=0.2),
])


def jacobian_frobenius(model, x):
    model.eval()
    x = x.clone().detach().requires_grad_(True)

    y = model(x)

    v = torch.randn_like(y)

    Jv = torch.autograd.grad(
        outputs=y,
        inputs=x,
        grad_outputs=v,
        create_graph=False,
        retain_graph=False
    )[0]

    # Proper per-sample Frobenius estimate
    per_sample = Jv.pow(2).flatten(1).sum(dim=1)

    return torch.sqrt(per_sample.mean())


def compute_metrics(encoder, loader):
    encoder = encoder.to(device)
    encoder.eval()

    jac_vals = []
    aug_vals = []
    noise_vals = []

    for i, (x, _) in enumerate(loader):
        if i == 5:  # limit batches
            break

        x = x.to(device)

        # ---- Jacobian ----
        jac_vals.append(jacobian_frobenius(encoder, x).item())

        # ---- Augmentation Sensitivity ----
        x_aug = torch.stack([
            tensor_aug(img.cpu()) for img in x
        ]).to(device)

        with torch.no_grad():
            f1 = encoder(x)
            f2 = encoder(x_aug)
            aug_vals.append(torch.norm(f1 - f2, dim=1).mean().item())

        # ---- Noise Sensitivity ----
        noise = torch.randn_like(x) * 0.1

        with torch.no_grad():
            f1 = encoder(x)
            f2 = encoder(x + noise)
            noise_vals.append(torch.norm(f1 - f2, dim=1).mean().item())

    return (
        np.mean(jac_vals),
        np.mean(aug_vals),
        np.mean(noise_vals)
    )

In [10]:
# =========================================================
# Supervised Data (for Linear Probe)
# =========================================================

supervised_train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=False,
    transform=T.ToTensor()
)

supervised_test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=False,
    transform=T.ToTensor()
)

supervised_train_loader = DataLoader(
    supervised_train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2
)

supervised_test_loader = DataLoader(
    supervised_test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2
)

print("Supervised loaders ready.")

Supervised loaders ready.


In [11]:
import os

# =========================================================
# CONFIG
# =========================================================

csv_path = "/kaggle/working/byol_simclr_results.csv"
target_seeds = [8, ]   # <-- change to range(10) when ready

# =========================================================
# RESUME LOGIC
# =========================================================

if os.path.exists(csv_path):
    existing_df = pd.read_csv(csv_path)
    completed_seeds = set(existing_df["Seed"].unique())
    all_results = existing_df.to_dict("records")
    print("Loaded existing results. Completed seeds:", completed_seeds)
else:
    completed_seeds = set()
    all_results = []
    print("No existing CSV found. Starting fresh.")

seeds = [s for s in target_seeds if s not in completed_seeds]

print("Seeds to run now:", seeds)

global_start = time.time()

# =========================================================
# MAIN LOOP
# =========================================================

for seed in seeds:

    print("\n====================================")
    print(f"Running Seed: {seed}")
    print("====================================")

    seed_start = time.time()
    set_seed(seed)

    # ------------------------
    # Train Models
    # ------------------------

    byol_encoder = train_byol(BYOL(), epochs=100)

    simclr_encoder = train_simclr(
        SimCLR(),
        epochs=100,
        lambda_jac=0.0
    )

    simclr_jac_encoder = train_simclr(
        SimCLR(),
        epochs=100,
        lambda_jac=1e-3
    )

    # ------------------------
    # Compute Metrics
    # ------------------------

    byol_metrics = compute_metrics(byol_encoder, test_loader)
    simclr_metrics = compute_metrics(simclr_encoder, test_loader)
    simclr_jac_metrics = compute_metrics(simclr_jac_encoder, test_loader)

    # ------------------------
    # Linear Probe
    # ------------------------

    byol_acc = train_probe(
        byol_encoder,
        supervised_train_loader,
        supervised_test_loader
    )

    simclr_acc = train_probe(
        simclr_encoder,
        supervised_train_loader,
        supervised_test_loader
    )

    simclr_jac_acc = train_probe(
        simclr_jac_encoder,
        supervised_train_loader,
        supervised_test_loader
    )

    # ------------------------
    # Store Results
    # ------------------------

    seed_results = [
        {
            "Seed": seed,
            "Method": "BYOL",
            "Jacobian": byol_metrics[0],
            "Aug_Sensitivity": byol_metrics[1],
            "Noise_Sensitivity": byol_metrics[2],
            "Linear_Probe_Acc": byol_acc
        },
        {
            "Seed": seed,
            "Method": "SimCLR",
            "Jacobian": simclr_metrics[0],
            "Aug_Sensitivity": simclr_metrics[1],
            "Noise_Sensitivity": simclr_metrics[2],
            "Linear_Probe_Acc": simclr_acc
        },
        {
            "Seed": seed,
            "Method": "SimCLR+Jac",
            "Jacobian": simclr_jac_metrics[0],
            "Aug_Sensitivity": simclr_jac_metrics[1],
            "Noise_Sensitivity": simclr_jac_metrics[2],
            "Linear_Probe_Acc": simclr_jac_acc
        }
    ]

    all_results.extend(seed_results)

    # ------------------------
    # SAVE AFTER EACH SEED
    # ------------------------

    results_df = pd.DataFrame(all_results)
    results_df.to_csv(csv_path, index=False)

    print("Saved results to:", csv_path)

    # ------------------------
    # Running Summary
    # ------------------------

    summary = results_df.groupby("Method").mean(numeric_only=True)
    print("\nRunning Mean Summary:")
    print(summary)

    # ------------------------
    # GPU Cleanup
    # ------------------------

    del byol_encoder, simclr_encoder, simclr_jac_encoder
    torch.cuda.empty_cache()

    seed_time = (time.time() - seed_start) / 60
    print(f"\nSeed {seed} finished in {seed_time:.2f} minutes")

# =========================================================
# FINAL SUMMARY
# =========================================================

final_df = pd.read_csv(csv_path)

print("\n================ FINAL RESULTS ================")
print(final_df)

total_time = (time.time() - global_start) / 60
print(f"\nTotal experiment time (this session): {total_time:.2f} minutes")

No existing CSV found. Starting fresh.
Seeds to run now: [8]

Running Seed: 8
[BYOL] 001/100 | Loss: 0.7552 | Epoch Time: 0.79m | Total Time: 0.79m
[BYOL] 002/100 | Loss: 0.5824 | Epoch Time: 0.77m | Total Time: 1.56m
[BYOL] 003/100 | Loss: 0.5089 | Epoch Time: 0.76m | Total Time: 2.33m
[BYOL] 004/100 | Loss: 0.4861 | Epoch Time: 0.75m | Total Time: 3.08m
[BYOL] 005/100 | Loss: 0.4743 | Epoch Time: 0.76m | Total Time: 3.84m
[BYOL] 006/100 | Loss: 0.4576 | Epoch Time: 0.75m | Total Time: 4.60m
[BYOL] 007/100 | Loss: 0.4474 | Epoch Time: 0.76m | Total Time: 5.36m
[BYOL] 008/100 | Loss: 0.4374 | Epoch Time: 0.78m | Total Time: 6.14m
[BYOL] 009/100 | Loss: 0.4331 | Epoch Time: 0.76m | Total Time: 6.90m
[BYOL] 010/100 | Loss: 0.4279 | Epoch Time: 0.76m | Total Time: 7.66m
[BYOL] 011/100 | Loss: 0.4204 | Epoch Time: 0.76m | Total Time: 8.42m
[BYOL] 012/100 | Loss: 0.4189 | Epoch Time: 0.76m | Total Time: 9.18m
[BYOL] 013/100 | Loss: 0.4138 | Epoch Time: 0.78m | Total Time: 9.96m
[BYOL] 014/1